# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-13_agentic_boundary.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker — see the [course README](https://github.com/justkrismanohar/fine-tuning-build/blob/main/README.md).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day3/T1-13_agentic_boundary.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day3"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day3" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


Ticket `TrainSession` needs CUDA. Tool-gate cells are ordinary Python and still run on CPU.


# T1-13 Agentic boundary + ticket routing (Track A)

**Time:** 13:30–14:15 &nbsp;|&nbsp; **Evidence:** ticket scoreboard + allow/ban list

23 training rows is **very small**. A weak result is a finding, not a failure of the lab.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** change only lines marked `# [CHANGE]`. Allowed values are named in the cell above.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path.

Git is not required. If you are stuck, load `completed/` for **this** notebook only and mark it as loaded from reference.


**Data ban:** CivicDesk fiction only. `heldout/cases.py` is a **test** set — do not train on it.


## 0. Backend


In [ ]:
# Colab track: demo/RECORDED. Free T4 may still run TrainSession if CUDA is free.
BACKEND = "demo"  # "olmo" | "phi" | "demo"

import civicdesk_ft as ft

cfg = ft.set_backend(BACKEND)


## 1. Base on the 9 held-out tickets, live

What this cell does: pre-flight tickets (or reuse today's file), then show T1 and T8 live — not only the card.


In [ ]:
# [RUN]
import civicdesk_ft as ft

record = ft.run_preflight(["tickets"])
ft.show_failure("T1")
print()
# T8 is injection — show from live file if present
from pathlib import Path
import json

pf = Path("logs/preflight_latest.json")
if pf.is_file():
    blob = json.loads(pf.read_text(encoding="utf-8"))
    raw = (blob.get("outputs") or {}).get("tickets", {}).get("T8")
    if raw and blob.get("measured"):
        print("LIVE, this session — T8")
        print(raw)
    else:
        print("T8 not in this session's measured pre-flight; open heldout/cases.py for the gold.")


## 2. Ticket training mix


In [ ]:
# [RUN]
import runpy
from pathlib import Path
import civicdesk_ft as ft

runpy.run_path(str(Path("build_tickets_set.py")), run_name="__main__")
ft.inspect_ticket_rows()


## 3. Train a ticket adapter (small data on purpose)


In [ ]:
# [RUN]
import ft_train

TARGETS = ["q_proj", "v_proj"]
RANK = 8
try:
    session = ft_train.TrainSession(task="tickets", targets=TARGETS, rank=RANK)
    session.summary()
    session.show_example(0)
except RuntimeError as exc:
    session = None
    print("SKIP train:", exc)


In [ ]:
# [RUN] one update; rerun to continue
if session is not None:
    session.step(1)


In [ ]:
# [CHANGE] more updates (e.g. 15). Small data — do not expect miracles.
MORE = 15
if session is not None:
    session.step(MORE)
    session.history()
    session.validate()
    session.save("adapters/my_tickets_run")


## 4. Score all 9 held-out tickets


In [ ]:
# [CHANGE]
ADAPTER = "adapters/my_tickets_run"  # or "prebuilt"

import ft_train

try:
    model = ft_train.load_adapter(ADAPTER, task="tickets")
    compare = ft_train.compare_live(model, "tickets")
    ft_train.ticket_scoreboard(compare)
except Exception as exc:
    print("Live tickets compare unavailable:", exc)
    try:
        compare = ft_train.load_recorded("tickets")
        ft_train.ticket_scoreboard(compare)
    except Exception as exc2:
        print(exc2)


## 5. Tool gate (rule code, not model behaviour)

The gate is ordinary Python. The model may *propose* an action; the gate decides.


In [ ]:
# [RUN]
import civicdesk_ft as ft

for action in [
    "draft_ticket",
    "quote_handbook",
    "draft_redacted_copy",
    "reset_password",
    "write_glpi",
    "award_contract",
    "scrape_web_for_negotiation",
    "release_to_panel",
    "auto_release_if_pass_checks",
]:
    print(f"{action:32} {ft.tool_gate(action)}")


In [ ]:
# [RUN] model proposes; gate decides
PROPOSED = "reset_password"  # pretend the ticket adapter suggested this
import civicdesk_ft as ft

print("model proposed:", PROPOSED)
print("tool_gate:", ft.tool_gate(PROPOSED))
print("The boundary is the gate, not the LoRA.")


In [ ]:
# [CHANGE] try grant_ad_rights, publish_notice, deploy_patrol
ACTION = "grant_ad_rights"
import civicdesk_ft as ft

print(ACTION, ft.tool_gate(ACTION))
